# Leakage Analysis — Sentetik Attitude × `Converted`

Bu notebook, `scripts/leakage_analysis.py` ile **aynı** `datagen.leakage` modülünü kullanır; sayısal sonuçlar birebir aynıdır. Notebook görselleştirme + yorum tarafını ekler.

Bakılan sorular:
1. attitude tek başına `Converted`'i ne kadar iyi tahmin ediyor? (AUC çok yüksekse leakage var)
2. attitude × `Converted` korelasyonu gerçekçi bantta mı? (Cramér's V ~0.20–0.40)
3. Her attitude sınıfında hem dönüşen hem dönüşmeyen lead var mı?
4. Ham veri tarafında sonradan dolan / outcome sızdıran kolonlar (`Tags`, `Lead Quality`, `Last Notable Activity`) ne kadar tehlikeli?

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from lead_priority.settings import REPO_ROOT
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
from datagen.leakage import compute_leakage_report

INTERACTIONS = REPO_ROOT / 'data' / 'synthetic' / 'interactions.parquet'
RAW = REPO_ROOT / 'data' / 'Lead Scoring.csv'

interactions = pd.read_parquet(INTERACTIONS)
interactions = interactions[interactions['text'].str.len() > 0].reset_index(drop=True)
raw = pd.read_csv(RAW)
print(f'interactions: {len(interactions)} rows')
print(f'raw         : {len(raw)} rows')
interactions.head()

In [ ]:
report = compute_leakage_report(interactions, raw)
print(f'attitude→Converted AUC (5-fold CV): {report.attitude_to_converted_auc:.3f}')
print(f"Cramér's V (attitude, Converted)   : {report.cramers_v:.3f}")
print(f'Mutual information (nats)          : {report.mutual_information_nats:.4f}')

## Attitude dağılımı

In [ ]:
dist = pd.Series(report.attitude_distribution).sort_values(ascending=False)
ax = dist.plot.bar(color='#4c72b0', figsize=(7, 3.5))
ax.set_ylabel('lead count')
ax.set_title('Attitude class distribution')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()
dist

## Crosstab: attitude × Converted

Her satırda hem dönüşen hem dönüşmeyen lead bulunmalı — aksi halde attitude `Converted`'in deterministik fonksiyonu demektir.

In [ ]:
crosstab_df = pd.DataFrame(report.crosstab).set_index('attitude')
crosstab_df

In [ ]:
pct_cols = [c for c in crosstab_df.columns if c.endswith('_pct')]
ax = crosstab_df[pct_cols].plot.bar(stacked=True, figsize=(7, 3.8), color=['#dd8452', '#4c72b0'])
ax.set_ylabel('% within attitude')
ax.set_title('Conversion rate within each attitude class')
ax.legend(['converted=0', 'converted=1'], loc='upper right')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Tabular leakage adayları

X Education tabular setinde **scoring modeline girmemesi gereken** kolonlar. Bunlar genellikle satış temsilcisinin lead'in akıbeti belli olduktan SONRA güncellediği alanlardır (`Tags=Closed by Horizzon`, `Lead Quality=High in Relevance` vb.). Tek bir kolonu LR'a vermek bile çok yüksek AUC veriyorsa ⇒ feature seçiminde drop edilmeli.

In [ ]:
suspects = pd.DataFrame(report.tabular_leakage_suspects)
suspects

In [ ]:
ax = suspects.set_index('column')['auc'].plot.barh(color='#c44e52', figsize=(7, 3))
ax.axvline(0.85, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('single-column AUC')
ax.set_title('Outcome-leaking column suspects (dashed line: 0.85 alarm)')
plt.tight_layout()
plt.show()

## Otomatik yorum

`compute_leakage_report` aşağıdaki bayrakları üretir. Hepsi sadece bilgi amaçlı — kararı insan verir.

In [ ]:
for note in report.interpretation:
    print('•', note)